# TabPFN variants

Every other model adds nothing to the TabPFN + GP stack (out-of-fold 0.7800): SVR, XGBoost, Extra Trees, MLP and Lasso all change it by less than ±0.0005. They all see the same 216 kNN-filled features, and their errors are 0.79–0.88 correlated with TabPFN's.

So this notebook gives TabPFN **different views of the data**, each a candidate column for the stack:

| Variant | Rows | Features | Missing values |
|---|---|---|---|
| `tabpfn` (reference, `TabPFN.ipynb`) | 1152 | the 216 selected, log on 11 | kNN fill |
| `tabpfn_nan` | 1152 | the 216 selected, log on 11 | **left as NaN** (TabPFN handles them) |
| `tabpfn_nan_300` | 1152 | **top 300** by \|Spearman ρ\| | left as NaN |
| `tabpfn_nan_all` | 1152 | **all 828** non-constant | left as NaN |
| `tabpfn_seeds` | 1152 | as the reference | kNN fill, **average of several `random_state`** |

Same 1152 rows (outliers removed), same `y`, same shared folds as every other model.

**Order:** screening with one seed (cell 4) → pick what helps the stack → final out-of-fold and test files with several seeds (cell 5) → one line in `MODELS` of `Stacking/stacking.ipynb`.

In [1]:
%pip install -q "tabpfn==9.2.0"

In [2]:
from pathlib import Path
import os

Path("/content/TabPFN").mkdir(exist_ok=True)
os.chdir("/content/TabPFN")

print("Dossier de travail :", os.getcwd())

Dossier de travail : /content/TabPFN


In [8]:
## Data

import os
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from tabpfn import TabPFNRegressor

from google.colab import userdata
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")   # ⚠️ put "PASTE_YOUR_KEY_HERE" back before any commit

X_knn = pd.read_csv("../X_train_log_knn.csv", index_col="id")          # the reference input: 1152 rows × 216
X_test_knn = pd.read_csv("../X_test_log_knn.csv", index_col="id")
y = pd.read_csv("../y_train.csv", index_col="id").loc[X_knn.index, "y"]
fold = pd.read_csv("../Stacking/folds.csv", index_col="id").loc[X_knn.index, "fold"]

# raw data, NaNs kept: same 1152 rows, constant columns dropped (as in data_prep.ipynb)
CONSTANT = ["x104", "x129", "x489", "x530"]
LOG_COLS = ["x115", "x159", "x415", "x458", "x507", "x531", "x542", "x641", "x665", "x702", "x742"]   # from data_prep.ipynb
raw = pd.read_csv("../X_train.csv", index_col="id").loc[X_knn.index].drop(columns=CONSTANT)
raw_test = pd.read_csv("../X_test.csv", index_col="id").drop(columns=CONSTANT)
raw[LOG_COLS], raw_test[LOG_COLS] = np.log(raw[LOG_COLS]), np.log(raw_test[LOG_COLS])   # log(NaN) stays NaN

print(f"knn {X_knn.shape} | raw {raw.shape} (NaN {raw.isna().mean().mean():.1%}) | test {raw_test.shape}")

knn (1152, 216) | raw (1152, 828) (NaN 7.6%) | test (776, 828)


In [5]:
## Feature sets

# |Spearman ρ| of every feature with the age, NaNs skipped (the same ranking as the 216-feature selection)
rho = raw.apply(lambda col: col.corr(y, method="spearman")).abs().sort_values(ascending=False)

SELECTED = list(X_knn.columns)
VARIANTS = {   # name -> (training features, test features); the heaviest last
    "tabpfn_nan":     (raw[SELECTED], raw_test[SELECTED]),
    "tabpfn_seeds":   (X_knn, X_test_knn),
    "tabpfn_nan_300": (raw[rho.index[:300]], raw_test[rho.index[:300]]),
    "tabpfn_nan_all": (raw, raw_test),
}
for name, (X, X_test) in VARIANTS.items():
    print(f"{name:15s}: {X.shape[1]} features, NaN {X.isna().mean().mean():.1%}")
print(f"the 216 selected = the top 216 by |ρ|: {set(SELECTED) == set(rho.index[:216])}")

tabpfn_nan     : 216 features, NaN 7.6%
tabpfn_seeds   : 216 features, NaN 0.0%
tabpfn_nan_300 : 300 features, NaN 7.6%
tabpfn_nan_all : 828 features, NaN 7.6%
the 216 selected = the top 216 by |ρ|: True


In [6]:
## Helpers

DEVICE = "cuda"   # the M1's GPU (MPS) runs out of memory; on a Mac with more memory, "auto" is much faster

def tabpfn_predict(X_train, y_train, X_pred, seeds):
    # average of one TabPFN per random_state
    return np.mean([TabPFNRegressor(random_state=s, device=DEVICE).fit(X_train, y_train).predict(X_pred) for s in seeds], axis=0)

def out_of_fold(X, seeds):
    oof = pd.Series(np.nan, index=X.index)
    for f in range(5):
        train, val = fold != f, fold == f
        oof[val] = tabpfn_predict(X[train], y[train], X[val], seeds)
        print(f"  fold {f} done", flush=True)
    return oof

# the current stack's out-of-fold predictions
stack_oof = pd.concat({m: pd.read_csv(f"../Stacking/oof_{m}.csv", index_col="id").iloc[:, 0]
                       for m in ["tabpfn", "gp"]}, axis=1).loc[X_knn.index]

def stack_r2(columns):
    # linear stacking scored on the shared folds, as in Stacking/stacking.ipynb
    pred = pd.Series(np.nan, index=columns.index)
    for f in range(5):
        train, val = fold != f, fold == f
        pred[val] = LinearRegression().fit(columns[train], y[train]).predict(columns[val])
    return r2_score(y, pred)

print(f"reference: TabPFN alone {r2_score(y, stack_oof['tabpfn']):.4f} | stack TabPFN + GP {stack_r2(stack_oof):.4f}")

reference: TabPFN alone 0.7772 | stack TabPFN + GP 0.7800


In [9]:
## Screening: one seed per variant (the seeds variant uses 3)

screen = {}
for name, (X, X_test) in VARIANTS.items():
    start = time.time()
    oof = out_of_fold(X, seeds=[0, 1, 2] if name == "tabpfn_seeds" else [0])
    screen[name] = oof
    candidates = stack_oof.assign(**{name: oof})
    print(f"{name:15s}: alone {r2_score(y, oof):.4f} | error corr. with TabPFN {(oof - y).corr(stack_oof['tabpfn'] - y):.3f} | "
          f"stack TabPFN + GP + it {stack_r2(candidates):.4f} | ({time.time() - start:.0f} s)")

# the seeds variant is a replacement for the TabPFN column, not an extra one
print(f"\nstack with tabpfn_seeds instead of tabpfn: {stack_r2(stack_oof.assign(tabpfn=screen['tabpfn_seeds'])):.4f}")

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
tabpfn_nan     : alone 0.7784 | error corr. with TabPFN 0.985 | stack TabPFN + GP + it 0.7823 | (40 s)
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
tabpfn_seeds   : alone 0.7783 | error corr. with TabPFN 0.997 | stack TabPFN + GP + it 0.7801 | (72 s)
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
tabpfn_nan_300 : alone 0.7797 | error corr. with TabPFN 0.976 | stack TabPFN + GP + it 0.7825 | (33 s)
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
tabpfn_nan_all : alone 0.7661 | error corr. with TabPFN 0.974 | stack TabPFN + GP + it 0.7789 | (83 s)

stack with tabpfn_seeds instead of tabpfn: 0.7807


**How to read it:** the reference stack (TabPFN + GP) is 0.7800. A variant is worth keeping if the stack with it gains more than about 0.002; differences below that are noise. A low error correlation with TabPFN is what makes a variant useful.

Then set `KEEP` below to the variants that helped, and run the last cell.

In [11]:
## Final files for the kept variants: out-of-fold + test predictions, average of SEEDS

KEEP = ["tabpfn_nan_300"]      # chosen from the screening: nan_300 + GP is the best stack (0.7829)
SEEDS = [0]                    # extra seeds did not help (tabpfn_seeds: +0.0001)

for name in KEEP:
    X, X_test = VARIANTS[name]
    oof = out_of_fold(X, SEEDS).rename(name)
    oof.to_csv(f"../Stacking/oof_{name}.csv")
    pd.Series(tabpfn_predict(X, y, X_test, SEEDS), index=X_test.index, name="y").to_csv(f"submission_{name}.csv")
    print(f"{name}: out-of-fold R2 {r2_score(y, oof):.4f} -> add to MODELS: "
          f'"{name}": ROOT / "TabPFN" / "submission_{name}.csv",')

  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
tabpfn_nan_300: out-of-fold R2 0.7797 -> add to MODELS: "tabpfn_nan_300": ROOT / "TabPFN" / "submission_tabpfn_nan_300.csv",


In [10]:
## Combinations of the useful variants (no new TabPFN runs)

combos = {
    "TabPFN + GP (current)":                    stack_oof,
    "+ nan":                                    stack_oof.assign(tabpfn_nan=screen["tabpfn_nan"]),
    "+ nan_300":                                stack_oof.assign(tabpfn_nan_300=screen["tabpfn_nan_300"]),
    "+ nan + nan_300":                          stack_oof.assign(tabpfn_nan=screen["tabpfn_nan"], tabpfn_nan_300=screen["tabpfn_nan_300"]),
    "nan_300 + GP (replacing TabPFN)":          stack_oof.drop(columns="tabpfn").assign(tabpfn_nan_300=screen["tabpfn_nan_300"]),
    "nan + nan_300 + GP (replacing TabPFN)":    stack_oof.drop(columns="tabpfn").assign(tabpfn_nan=screen["tabpfn_nan"], tabpfn_nan_300=screen["tabpfn_nan_300"]),
}
for name, cols in combos.items():
    print(f"{name:40s}: stack R2 {stack_r2(cols):.4f}")


TabPFN + GP (current)                   : stack R2 0.7800
+ nan                                   : stack R2 0.7823
+ nan_300                               : stack R2 0.7825
+ nan + nan_300                         : stack R2 0.7818
nan_300 + GP (replacing TabPFN)         : stack R2 0.7829
nan + nan_300 + GP (replacing TabPFN)   : stack R2 0.7817


In [12]:
import gc
import torch

# Même variante, mêmes observations et mêmes folds
X_tune = VARIANTS["tabpfn_nan_300"][0]
reference_oof = screen["tabpfn_nan_300"].loc[X_tune.index].copy()
gp_oof = stack_oof["gp"].loc[X_tune.index]

# Identifier le nombre de membres réellement utilisé par "auto"
first_fold = sorted(fold.unique())[0]
probe_train = fold != first_fold

probe = TabPFNRegressor(
    random_state=0,
    device=DEVICE,
    n_estimators="auto"
).fit(X_tune.loc[probe_train], y.loc[probe_train])

default_n = int(probe.n_estimators_)
print(f"Nombre de membres utilisé par la référence : {default_n}")

del probe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# Évaluer uniquement le stack variante + GP
def score_with_gp(predictions):
    columns = pd.DataFrame({
        "gp": gp_oof,
        "tabpfn_nan_300": predictions
    })
    return stack_r2(columns)


reference_stack_score = score_with_gp(reference_oof)

# Conserver toutes les OOF pour les comparaisons suivantes
n_estimators_oof = {default_n: reference_oof}

n_estimators_records = [{
    "n_estimators": default_n,
    "configuration": "Référence auto",
    "R2_TabPFN": r2_score(y, reference_oof),
    "R2_stack_GP": reference_stack_score,
    "gain_stack": 0.0,
    "secondes": np.nan
}]

# Ne pas recalculer le nombre déjà utilisé par la référence
counts_to_test = [
    n for n in [8, 16, 32]
    if n != default_n
]

for n in counts_to_test:
    start = time.time()
    predictions_oof = pd.Series(np.nan, index=X_tune.index)

    for f in sorted(fold.unique()):
        train = fold != f
        valid = fold == f

        model_tune = TabPFNRegressor(
            n_estimators=n,
            random_state=0,
            device=DEVICE
        )

        model_tune.fit(
            X_tune.loc[train],
            y.loc[train]
        )

        predictions_oof.loc[valid] = model_tune.predict(
            X_tune.loc[valid]
        )

        print(f"n_estimators={n} | fold {f} terminé", flush=True)

        # Libérer les modèles entre les évaluations
        del model_tune
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    n_estimators_oof[n] = predictions_oof.copy()

    stack_score = score_with_gp(predictions_oof)

    n_estimators_records.append({
        "n_estimators": n,
        "configuration": "Valeur explicite",
        "R2_TabPFN": r2_score(y, predictions_oof),
        "R2_stack_GP": stack_score,
        "gain_stack": stack_score - reference_stack_score,
        "secondes": time.time() - start
    })

    print(
        f"→ {n} membres : "
        f"TabPFN {r2_score(y, predictions_oof):.5f} | "
        f"stack {stack_score:.5f} | "
        f"gain {stack_score - reference_stack_score:+.5f}\n",
        flush=True
    )

# Résumé
n_estimators_results = (
    pd.DataFrame(n_estimators_records)
    .set_index("n_estimators")
    .sort_values("R2_stack_GP", ascending=False)
)

display(n_estimators_results.round(5))

Nombre de membres utilisé par la référence : 8
n_estimators=16 | fold 0 terminé
n_estimators=16 | fold 1 terminé
n_estimators=16 | fold 2 terminé
n_estimators=16 | fold 3 terminé
n_estimators=16 | fold 4 terminé
→ 16 membres : TabPFN 0.77798 | stack 0.78169 | gain -0.00121

n_estimators=32 | fold 0 terminé
n_estimators=32 | fold 1 terminé
n_estimators=32 | fold 2 terminé
n_estimators=32 | fold 3 terminé
n_estimators=32 | fold 4 terminé
→ 32 membres : TabPFN 0.77873 | stack 0.78261 | gain -0.00028



,configuration,R2_TabPFN,R2_stack_GP,gain_stack,secondes
n_estimators,,,,,
8,Référence auto,0.77966,0.78290,0.00000,NaN
32,Valeur explicite,0.77873,0.78261,-0.00028,129.52840
16,Valeur explicite,0.77798,0.78169,-0.00121,67.24098


In [13]:


X_temperature = VARIANTS["tabpfn_nan_300"][0]

# Référence : température automatique et agrégation par défaut
reference_temperature_oof = screen["tabpfn_nan_300"].loc[
    X_temperature.index
].copy()

reference_temperature_stack = score_with_gp(
    reference_temperature_oof
)

# Conserver les prédictions pour examiner les meilleurs essais ensuite
temperature_oof = {
    ("auto", False): reference_temperature_oof
}

temperature_records = [{
    "temperature": "auto",
    "average_before_softmax": False,
    "R2_TabPFN": r2_score(y, reference_temperature_oof),
    "R2_stack_GP": reference_temperature_stack,
    "gain_stack": 0.0,
    "secondes": np.nan
}]

# La référence ("auto", False) est déjà disponible
temperature_configs = [
    (temperature, aggregation)
    for temperature in ["auto", 0.7, 1.0, 1.3]
    for aggregation in [False, True]
    if (temperature, aggregation) != ("auto", False)
]

print(
    f"Référence du stack : {reference_temperature_stack:.5f}",
    flush=True
)

for temperature, aggregation in temperature_configs:
    start = time.time()
    predictions_oof = pd.Series(
        np.nan, index=X_temperature.index
    )

    for f in sorted(fold.unique()):
        train = fold != f
        valid = fold == f

        model_temperature = TabPFNRegressor(
            n_estimators=8,
            random_state=0,
            device=DEVICE,
            softmax_temperature=temperature,
            average_before_softmax=aggregation
        )

        model_temperature.fit(
            X_temperature.loc[train],
            y.loc[train]
        )

        predictions_oof.loc[valid] = model_temperature.predict(
            X_temperature.loc[valid]
        )

        print(
            f"T={temperature} | "
            f"average_before_softmax={aggregation} | "
            f"fold {f} terminé",
            flush=True
        )

        del model_temperature
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    temperature_oof[(temperature, aggregation)] = (
        predictions_oof.copy()
    )

    individual_score = r2_score(y, predictions_oof)
    stack_score = score_with_gp(predictions_oof)

    temperature_records.append({
        "temperature": temperature,
        "average_before_softmax": aggregation,
        "R2_TabPFN": individual_score,
        "R2_stack_GP": stack_score,
        "gain_stack": stack_score - reference_temperature_stack,
        "secondes": time.time() - start
    })

    print(
        f"→ TabPFN {individual_score:.5f} | "
        f"stack {stack_score:.5f} | "
        f"gain {stack_score - reference_temperature_stack:+.5f}\n",
        flush=True
    )

# Tableau final, référence incluse
temperature_results = (
    pd.DataFrame(temperature_records)
    .set_index(["temperature", "average_before_softmax"])
    .sort_values("R2_stack_GP", ascending=False)
)

display(temperature_results.round(5))

Référence du stack : 0.78290
T=auto | average_before_softmax=True | fold 0 terminé
T=auto | average_before_softmax=True | fold 1 terminé
T=auto | average_before_softmax=True | fold 2 terminé
T=auto | average_before_softmax=True | fold 3 terminé
T=auto | average_before_softmax=True | fold 4 terminé
→ TabPFN 0.78072 | stack 0.78374 | gain +0.00084

T=0.7 | average_before_softmax=False | fold 0 terminé
T=0.7 | average_before_softmax=False | fold 1 terminé
T=0.7 | average_before_softmax=False | fold 2 terminé
T=0.7 | average_before_softmax=False | fold 3 terminé
T=0.7 | average_before_softmax=False | fold 4 terminé
→ TabPFN 0.77333 | stack 0.78110 | gain -0.00180

T=0.7 | average_before_softmax=True | fold 0 terminé
T=0.7 | average_before_softmax=True | fold 1 terminé
T=0.7 | average_before_softmax=True | fold 2 terminé
T=0.7 | average_before_softmax=True | fold 3 terminé
T=0.7 | average_before_softmax=True | fold 4 terminé
→ TabPFN 0.77409 | stack 0.78159 | gain -0.00131

T=1.0 | average_

R2_TabPFN  R2_stack_GP  gain_stack  \
temperature average_before_softmax                                       
auto        True                      0.78072      0.78374     0.00084   
1.0         True                      0.78072      0.78374     0.00084   
1.3         True                      0.78135      0.78331     0.00042   
auto        False                     0.77966      0.78290     0.00000   
1.0         False                     0.77966      0.78290     0.00000   
1.3         False                     0.78040      0.78248    -0.00042   
0.7         True                      0.77409      0.78159    -0.00131   
            False                     0.77333      0.78110    -0.00180   

                                    secondes  
temperature average_before_softmax            
auto        True                    35.45537  
1.0         True                    34.39143  
1.3         True                    34.48504  
auto        False                        NaN  
1.0         False                   34.63246  
1.3         False                   34.67327  
0.7         True                    34.44748  
            False                   34.29552

In [14]:
# OOF déjà calculées : référence et configuration candidate
aggregation_candidates = {
    "Référence": temperature_oof[("auto", False)],
    "Agrégation True": temperature_oof[(1.0, True)]
}

# Prédictions de validation du stack pour chaque configuration
aggregation_stack_oof = pd.DataFrame(
    np.nan,
    index=y.index,
    columns=aggregation_candidates.keys()
)

for name, tabpfn_predictions in aggregation_candidates.items():
    columns = pd.DataFrame({
        "gp": stack_oof["gp"],
        "tabpfn": tabpfn_predictions
    }).loc[y.index]

    for f in sorted(fold.unique()):
        train = fold != f
        valid = fold == f

        meta_model = LinearRegression().fit(
            columns.loc[train],
            y.loc[train]
        )

        aggregation_stack_oof.loc[valid, name] = (
            meta_model.predict(columns.loc[valid])
        )

# Comparaison sur exactement les mêmes observations dans chaque fold
aggregation_fold_records = []

for f in sorted(fold.unique()):
    valid = fold == f

    baseline_score = r2_score(
        y.loc[valid],
        aggregation_stack_oof.loc[valid, "Référence"]
    )
    candidate_score = r2_score(
        y.loc[valid],
        aggregation_stack_oof.loc[valid, "Agrégation True"]
    )

    aggregation_fold_records.append({
        "fold": f,
        "R2_reference": baseline_score,
        "R2_candidate": candidate_score,
        "gain": candidate_score - baseline_score
    })

aggregation_fold_results = pd.DataFrame(
    aggregation_fold_records
).set_index("fold")

display(aggregation_fold_results.round(6))

print(
    "Folds améliorés : "
    f"{(aggregation_fold_results['gain'] > 0).sum()}"
    f"/{len(aggregation_fold_results)}"
)

for name in aggregation_stack_oof.columns:
    print(
        f"{name} — R² OOF global : "
        f"{r2_score(y, aggregation_stack_oof[name]):.6f}"
    )

,R2_reference,R2_candidate,gain
fold,,,
0,0.818312,0.819099,0.000787
1,0.799191,0.800560,0.001369
2,0.748545,0.748472,-0.000072
3,0.765020,0.766184,0.001164
4,0.779580,0.780511,0.000931


Folds améliorés : 4/5
Référence — R² OOF global : 0.782896
Agrégation True — R² OOF global : 0.783738


In [15]:
X_seed_test = VARIANTS["tabpfn_nan_300"][0]

# Seed 0 déjà calculé avec température = 1.0
seed_comparison_oof = {
    (0, False): temperature_oof[(1.0, False)].copy(),
    (0, True): temperature_oof[(1.0, True)].copy()
}

# Nouveaux essais : seuls le seed et l'agrégation changent
for seed in [1, 2]:
    for aggregation in [False, True]:
        start = time.time()
        predictions_oof = pd.Series(
            np.nan, index=X_seed_test.index
        )

        for f in sorted(fold.unique()):
            train = fold != f
            valid = fold == f

            model_seed = TabPFNRegressor(
                n_estimators=8,
                random_state=seed,
                device=DEVICE,
                softmax_temperature=1.0,
                average_before_softmax=aggregation
            )

            model_seed.fit(
                X_seed_test.loc[train],
                y.loc[train]
            )

            predictions_oof.loc[valid] = model_seed.predict(
                X_seed_test.loc[valid]
            )

            print(
                f"Seed {seed} | agrégation={aggregation} | "
                f"fold {f} terminé",
                flush=True
            )

            del model_seed
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        seed_comparison_oof[(seed, aggregation)] = (
            predictions_oof.copy()
        )

        print(
            f"→ TabPFN {r2_score(y, predictions_oof):.5f} | "
            f"stack GP {score_with_gp(predictions_oof):.5f} | "
            f"{time.time() - start:.0f} s\n",
            flush=True
        )


# Résultats des six configurations
seed_summary = pd.DataFrame([
    {
        "seed": seed,
        "aggregation": aggregation,
        "R2_TabPFN": r2_score(y, predictions),
        "R2_stack_GP": score_with_gp(predictions)
    }
    for (seed, aggregation), predictions
    in seed_comparison_oof.items()
])

display(
    seed_summary.set_index(
        ["seed", "aggregation"]
    ).sort_index().round(6)
)

# Comparaison True / False pour chaque seed
stack_by_seed = seed_summary.pivot(
    index="seed",
    columns="aggregation",
    values="R2_stack_GP"
).rename(columns={
    False: "False",
    True: "True"
})

stack_by_seed.columns.name = None
stack_by_seed["gain_True"] = (
    stack_by_seed["True"] - stack_by_seed["False"]
)

display(stack_by_seed.round(6))


# Moyenne des prédictions des trois seeds, pour chaque agrégation
seed_average_oof = {}

for aggregation in [False, True]:
    seed_average_oof[aggregation] = pd.concat(
        [
            seed_comparison_oof[(seed, aggregation)]
            for seed in [0, 1, 2]
        ],
        axis=1
    ).mean(axis=1)

seed_average_summary = pd.DataFrame([
    {
        "aggregation": aggregation,
        "R2_TabPFN_moyenne_3_seeds": r2_score(y, predictions),
        "R2_stack_GP": score_with_gp(predictions)
    }
    for aggregation, predictions in seed_average_oof.items()
]).set_index("aggregation")

print("\nMoyenne des trois seeds :")
display(seed_average_summary.round(6))

Seed 1 | agrégation=False | fold 0 terminé
Seed 1 | agrégation=False | fold 1 terminé
Seed 1 | agrégation=False | fold 2 terminé
Seed 1 | agrégation=False | fold 3 terminé
Seed 1 | agrégation=False | fold 4 terminé
→ TabPFN 0.77686 | stack GP 0.78156 | 36 s

Seed 1 | agrégation=True | fold 0 terminé
Seed 1 | agrégation=True | fold 1 terminé
Seed 1 | agrégation=True | fold 2 terminé
Seed 1 | agrégation=True | fold 3 terminé
Seed 1 | agrégation=True | fold 4 terminé
→ TabPFN 0.77811 | stack GP 0.78250 | 34 s

Seed 2 | agrégation=False | fold 0 terminé
Seed 2 | agrégation=False | fold 1 terminé
Seed 2 | agrégation=False | fold 2 terminé
Seed 2 | agrégation=False | fold 3 terminé
Seed 2 | agrégation=False | fold 4 terminé
→ TabPFN 0.77642 | stack GP 0.78068 | 34 s

Seed 2 | agrégation=True | fold 0 terminé
Seed 2 | agrégation=True | fold 1 terminé
Seed 2 | agrégation=True | fold 2 terminé
Seed 2 | agrégation=True | fold 3 terminé
Seed 2 | agrégation=True | fold 4 terminé
→ TabPFN 0.77721 |

R2_TabPFN  R2_stack_GP
seed aggregation                        
0    False         0.779664     0.782896
     True          0.780718     0.783738
1    False         0.776865     0.781563
     True          0.778107     0.782502
2    False         0.776416     0.780678
     True          0.777205     0.781273

,False,True,gain_True
seed,,,
0,0.782896,0.783738,0.000842
1,0.781563,0.782502,0.000939
2,0.780678,0.781273,0.000595



Moyenne des trois seeds :


,R2_TabPFN_moyenne_3_seeds,R2_stack_GP
aggregation,,
False,0.779203,0.782936
True,0.780265,0.783749


In [17]:
from pathlib import Path
from io import BytesIO
from google.colab import files
import json
import shutil

# Sélectionner sample.csv et submission_gp.csv sur ton ordinateur
uploaded = files.upload()

sample = pd.read_csv(BytesIO(uploaded["sample.csv"]))
gp_test = pd.read_csv(
    BytesIO(uploaded["submission_gp.csv"]),
    index_col="id"
)["y"]

# Configuration retenue
FINAL_PARAMS = {
    "n_estimators": 8,
    "softmax_temperature": 1.0,
    "average_before_softmax": True,
    "random_state": 0
}

X_final, X_test_final = VARIANTS["tabpfn_nan_300"]
y_final = y.loc[X_final.index]

# OOF correspondant exactement à cette configuration
tuned_oof = temperature_oof[(1.0, True)].loc[X_final.index].copy()
tuned_oof.name = "tabpfn_nan_300_tuned"

# Vérifier les identifiants avant de construire les submissions
assert sample["id"].is_unique and gp_test.index.is_unique
assert (
    set(sample["id"])
    == set(X_test_final.index)
    == set(gp_test.index)
), "Les identifiants test ne correspondent pas"

# Modèle final sur toutes les observations conservées
print("Calcul des prédictions test TabPFN...", flush=True)

final_tabpfn = TabPFNRegressor(
    **FINAL_PARAMS,
    device=DEVICE
)
final_tabpfn.fit(X_final, y_final)

tabpfn_test = pd.Series(
    final_tabpfn.predict(X_test_final),
    index=X_test_final.index,
    name="tabpfn_nan_300_tuned"
)

# Stack final : apprendre les coefficients sur toutes les OOF
meta_train = pd.DataFrame({
    "gp": stack_oof["gp"].loc[X_final.index],
    "tabpfn_nan_300_tuned": tuned_oof
})

final_meta = LinearRegression().fit(meta_train, y_final)

# Même ordre de modèles et même ordre d'IDs que sample.csv
meta_test = pd.DataFrame({
    "gp": gp_test,
    "tabpfn_nan_300_tuned": tabpfn_test
}).loc[sample["id"], meta_train.columns]

assert np.isfinite(meta_test.to_numpy()).all()

# Dossier distinct des fichiers existants
export_dir = Path("/content/tabpfn_tuned_results")
export_dir.mkdir(exist_ok=True)

# Submission TabPFN seul
submission_single = sample.copy()
submission_single["y"] = tabpfn_test.loc[sample["id"]].to_numpy()
submission_single.to_csv(
    export_dir / "submission_tabpfn_nan_300_tuned.csv",
    index=False
)

# Submission du stack TabPFN tuné + GP
submission_stack = sample.copy()
submission_stack["y"] = final_meta.predict(meta_test)
submission_stack.to_csv(
    export_dir / "submission_stack_tabpfn_tuned_gp.csv",
    index=False
)

# OOF utiles pour réutiliser la variante dans le projet
tuned_oof.to_csv(
    export_dir / "oof_tabpfn_nan_300_tuned.csv"
)

# Conserver les paramètres et les coefficients du stack
configuration = {
    "variant": "tabpfn_nan_300",
    "parameters": FINAL_PARAMS,
    "features": X_final.columns.tolist(),
    "stack_coefficients": dict(
        zip(meta_train.columns, final_meta.coef_.tolist())
    ),
    "stack_intercept": float(final_meta.intercept_)
}

(export_dir / "configuration.json").write_text(
    json.dumps(configuration, indent=2)
)

print("\nFichiers créés :")
for path in sorted(export_dir.iterdir()):
    print(path.name)

display(submission_stack.head())

# Télécharger tous les résultats dans une archive
archive = shutil.make_archive(
    "/content/tabpfn_tuned_results",
    "zip",
    root_dir=export_dir
)
files.download(archive)
from io import BytesIO
from google.colab import files
import json
import shutil

# Sélectionner sample.csv et submission_gp.csv sur ton ordinateur
uploaded = files.upload()

sample = pd.read_csv(BytesIO(uploaded["sample.csv"]))
gp_test = pd.read_csv(
    BytesIO(uploaded["submission_gp.csv"]),
    index_col="id"
)["y"]

# Configuration retenue
FINAL_PARAMS = {
    "n_estimators": 8,
    "softmax_temperature": 1.0,
    "average_before_softmax": True,
    "random_state": 0
}

X_final, X_test_final = VARIANTS["tabpfn_nan_300"]
y_final = y.loc[X_final.index]

# OOF correspondant exactement à cette configuration
tuned_oof = temperature_oof[(1.0, True)].loc[X_final.index].copy()
tuned_oof.name = "tabpfn_nan_300_tuned"

# Vérifier les identifiants avant de construire les submissions
assert sample["id"].is_unique and gp_test.index.is_unique
assert (
    set(sample["id"])
    == set(X_test_final.index)
    == set(gp_test.index)
), "Les identifiants test ne correspondent pas"

# Modèle final sur toutes les observations conservées
print("Calcul des prédictions test TabPFN...", flush=True)

final_tabpfn = TabPFNRegressor(
    **FINAL_PARAMS,
    device=DEVICE
)
final_tabpfn.fit(X_final, y_final)

tabpfn_test = pd.Series(
    final_tabpfn.predict(X_test_final),
    index=X_test_final.index,
    name="tabpfn_nan_300_tuned"
)

# Stack final : apprendre les coefficients sur toutes les OOF
meta_train = pd.DataFrame({
    "gp": stack_oof["gp"].loc[X_final.index],
    "tabpfn_nan_300_tuned": tuned_oof
})

final_meta = LinearRegression().fit(meta_train, y_final)

# Même ordre de modèles et même ordre d'IDs que sample.csv
meta_test = pd.DataFrame({
    "gp": gp_test,
    "tabpfn_nan_300_tuned": tabpfn_test
}).loc[sample["id"], meta_train.columns]

assert np.isfinite(meta_test.to_numpy()).all()

# Dossier distinct des fichiers existants
export_dir = Path("/content/tabpfn_tuned_results")
export_dir.mkdir(exist_ok=True)

# Submission TabPFN seul
submission_single = sample.copy()
submission_single["y"] = tabpfn_test.loc[sample["id"]].to_numpy()
submission_single.to_csv(
    export_dir / "submission_tabpfn_nan_300_tuned.csv",
    index=False
)

# Submission du stack TabPFN tuné + GP
submission_stack = sample.copy()
submission_stack["y"] = final_meta.predict(meta_test)
submission_stack.to_csv(
    export_dir / "submission_stack_tabpfn_tuned_gp.csv",
    index=False
)

# OOF utiles pour réutiliser la variante dans le projet
tuned_oof.to_csv(
    export_dir / "oof_tabpfn_nan_300_tuned.csv"
)

# Conserver les paramètres et les coefficients du stack
configuration = {
    "variant": "tabpfn_nan_300",
    "parameters": FINAL_PARAMS,
    "features": X_final.columns.tolist(),
    "stack_coefficients": dict(
        zip(meta_train.columns, final_meta.coef_.tolist())
    ),
    "stack_intercept": float(final_meta.intercept_)
}

(export_dir / "configuration.json").write_text(
    json.dumps(configuration, indent=2)
)

print("\nFichiers créés :")
for path in sorted(export_dir.iterdir()):
    print(path.name)

display(submission_stack.head())

# Télécharger tous les résultats dans une archive
archive = shutil.make_archive(
    "/content/tabpfn_tuned_results",
    "zip",
    root_dir=export_dir
)
files.download(archive)

Saving sample.csv to sample.csv
Saving submission_gp.csv to submission_gp.csv
Calcul des prédictions test TabPFN...

Fichiers créés :
configuration.json
oof_tabpfn_nan_300_tuned.csv
submission_stack_tabpfn_tuned_gp.csv
submission_tabpfn_nan_300_tuned.csv


,id,y
0,0.0,61.101828
1,1.0,75.919099
2,2.0,71.181326
3,3.0,75.621295
4,4.0,73.329460


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

KeyError: 'sample.csv'